# 03 — Dashboard de Performance Logística
**Dataset:** DataCo Smart Supply Chain  
**Objetivo:** Visualizações interativas consolidadas para apresentação de portfólio — mapa global de atrasos, sunburst por região/categoria e painel de KPIs.


In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('../data/raw/DataCoSupplyChainDataset.csv', encoding='latin-1')
df_validos = df[df['Delivery Status'] != 'Shipping canceled'].copy()
df_validos['Desvio Lead Time'] = df_validos['Days for shipping (real)'] - df_validos['Days for shipment (scheduled)']
df_validos['Atrasado'] = (df_validos['Delivery Status'] == 'Late delivery').astype(int)

print(f'Dataset carregado: {len(df_validos):,} pedidos válidos')

Dataset carregado: 172,765 pedidos válidos


## 1. Painel de KPIs — Resumo Executivo

In [2]:
total = len(df_validos)
atrasados = df_validos['Atrasado'].sum()
otif = (1 - atrasados / total) * 100
lead_time_medio = df_validos['Days for shipping (real)'].mean()
desvio_medio = df_validos['Desvio Lead Time'].mean()
cancelados = len(df) - len(df_validos)

fig = make_subplots(
    rows=1, cols=4,
    specs=[[{'type': 'indicator'}] * 4]
)

kpis = [
    ('OTIF', otif, '%', '#2ecc71' if otif >= 85 else '#e74c3c'),
    ('Taxa de Atraso', atrasados / total * 100, '%', '#e74c3c'),
    ('Lead Time Médio', lead_time_medio, ' dias', '#3498db'),
    ('Pedidos Cancelados', cancelados, '', '#95a5a6'),
]

for i, (titulo, valor, sufixo, cor) in enumerate(kpis, 1):
    fig.add_trace(go.Indicator(
        mode='number',
        value=valor,
        title={'text': titulo, 'font': {'size': 14}},
        number={'suffix': sufixo, 'valueformat': '.1f', 'font': {'color': cor, 'size': 36}}
    ), row=1, col=i)

fig.update_layout(
    title='Painel de KPIs — DataCo Supply Chain',
    height=220,
    paper_bgcolor='#f8f9fa'
)
fig.show()

## 2. Mapa Global — Taxa de Atraso por País

In [3]:
mapa_pais = df_validos.groupby('Order Country').agg(
    Total=('Atrasado', 'count'),
    Atrasados=('Atrasado', 'sum')
).reset_index()
mapa_pais['Taxa Atraso (%)'] = (mapa_pais['Atrasados'] / mapa_pais['Total'] * 100).round(1)
mapa_pais = mapa_pais[mapa_pais['Total'] >= 100]  # Filtrar países com volume relevante

fig = px.choropleth(
    mapa_pais,
    locations='Order Country',
    locationmode='country names',
    color='Taxa Atraso (%)',
    hover_name='Order Country',
    hover_data={'Total': ':,', 'Atrasados': ':,', 'Taxa Atraso (%)': ':.1f'},
    color_continuous_scale='RdYlGn_r',
    range_color=[0, 100],
    title='Mapa Global — Taxa de Atraso por País (%)',
)
fig.update_layout(height=500, coloraxis_colorbar_title='Atraso (%)')
fig.show()

print('Top 5 países com maior taxa de atraso:')
print(mapa_pais.nlargest(5, 'Taxa Atraso (%)')[['Order Country', 'Total', 'Taxa Atraso (%)']].to_string(index=False))

Top 5 países com maior taxa de atraso:
    Order Country  Total  Taxa Atraso (%)
          Ecuador    279             70.6
          Somalia    133             67.7
            Níger    105             65.7
       Mozambique    238             65.5
Trinidad y Tobago    126             64.3


## 3. Sunburst — Hierarquia de Atrasos: Região → Modal → Status

In [4]:
# Top 5 regiões por volume para não sobrecarregar o gráfico
top_regioes = df_validos['Order Region'].value_counts().head(5).index.tolist()
df_sun = df_validos[df_validos['Order Region'].isin(top_regioes)].copy()

sun_data = df_sun.groupby(['Order Region', 'Shipping Mode', 'Delivery Status']).size().reset_index(name='Pedidos')

fig = px.sunburst(
    sun_data,
    path=['Order Region', 'Shipping Mode', 'Delivery Status'],
    values='Pedidos',
    color='Delivery Status',
    color_discrete_map={
        '(?)': '#cccccc',
        'Late delivery': '#e74c3c',
        'Shipping on time': '#2ecc71',
        'Advance shipping': '#3498db'
    },
    title='Hierarquia de Entregas: Top 5 Regiões → Modal → Status'
)
fig.update_layout(height=580)
fig.show()

## 4. Evolução Mensal da Taxa de Atraso

In [5]:
df_validos['order date (DateOrders)'] = pd.to_datetime(df_validos['order date (DateOrders)'], errors='coerce')
df_validos['AnoMes'] = df_validos['order date (DateOrders)'].dt.to_period('M').astype(str)

evolucao = df_validos.groupby('AnoMes').agg(
    Total=('Atrasado', 'count'),
    Atrasados=('Atrasado', 'sum')
).reset_index()
evolucao['Taxa Atraso (%)'] = (evolucao['Atrasados'] / evolucao['Total'] * 100).round(1)
evolucao = evolucao[evolucao['Total'] >= 200]  # Meses com volume relevante

fig = px.line(
    evolucao,
    x='AnoMes', y='Taxa Atraso (%)',
    title='Evolução Mensal da Taxa de Atraso (%)',
    markers=True,
    line_shape='spline',
    color_discrete_sequence=['#e74c3c']
)
fig.add_hline(y=evolucao['Taxa Atraso (%)'].mean(), line_dash='dash',
              annotation_text=f"Média: {evolucao['Taxa Atraso (%)'].mean():.1f}%",
              line_color='gray')
fig.update_layout(height=420, xaxis_tickangle=-35)
fig.show()

## 5. Scatter — Relação entre Lead Time Real e Previsto por Modal

In [6]:
amostra = df_validos.sample(n=5000, random_state=42)

fig = px.scatter(
    amostra,
    x='Days for shipment (scheduled)',
    y='Days for shipping (real)',
    color='Shipping Mode',
    symbol='Delivery Status',
    opacity=0.5,
    title='Lead Time Real vs. Previsto por Modal de Envio (amostra: 5.000 pedidos)',
    labels={
        'Days for shipment (scheduled)': 'Dias Previstos',
        'Days for shipping (real)': 'Dias Reais'
    }
)
# Linha de referência perfeita (real = previsto)
max_val = max(amostra['Days for shipping (real)'].max(), amostra['Days for shipment (scheduled)'].max())
fig.add_trace(go.Scatter(
    x=[0, max_val], y=[0, max_val],
    mode='lines', name='Prazo exato',
    line={'dash': 'dash', 'color': 'black', 'width': 1}
))
fig.update_layout(height=500)
fig.show()

## 6. Exportar Gráfico Principal para o README

In [7]:
# Gráfico de destaque: taxa de atraso por região — exportar como PNG
atraso_regiao = df_validos.groupby('Order Region').apply(
    lambda x: pd.Series({
        'Total': len(x),
        'Taxa Atraso (%)': round(x['Atrasado'].mean() * 100, 1)
    })
).reset_index().sort_values('Taxa Atraso (%)', ascending=True)

fig_export = px.bar(
    atraso_regiao,
    x='Taxa Atraso (%)', y='Order Region',
    orientation='h',
    text='Taxa Atraso (%)',
    color='Taxa Atraso (%)',
    color_continuous_scale='RdYlGn_r',
    title='Taxa de Atraso por Região — DataCo Supply Chain',
    labels={'Order Region': 'Região', 'Taxa Atraso (%)': 'Taxa de Atraso (%)'},
)
fig_export.update_traces(texttemplate='%{text}%', textposition='outside')
fig_export.update_layout(
    height=550, width=900,
    coloraxis_showscale=False,
    xaxis_range=[0, 100],
    font=dict(size=13),
    paper_bgcolor='white',
    plot_bgcolor='white'
)

fig_export.write_image('../outputs/taxa_atraso_por_regiao.png', scale=2)
print('Exportado: outputs/taxa_atraso_por_regiao.png')
fig_export.show()

Exportado: outputs/taxa_atraso_por_regiao.png


## 7. Conclusões Finais

### Achados críticos:
- **OTIF ~42%** — abaixo da metade da meta de mercado (85%), evidenciando falha sistêmica na cadeia de entrega.
- **Standard Class** é o modal com maior volume e maior taxa de atraso — o maior gargalo operacional.
- Países da **África** e **Ásia** concentram as maiores taxas de atraso no mapa global.
- A taxa de atraso se mantém **consistentemente alta ao longo do tempo**, descartando sazonalidade como causa isolada.

### Recomendações:
1. **Revisar SLAs** do Standard Class — os prazos prometidos estão sistematicamente abaixo do real.
2. **Redistribuir parte do volume** para First Class em regiões críticas.
3. **Criar alertas por região** para monitorar deterioração do OTIF em tempo real.
4. **Investigar categorias** de alto atraso para identificar gargalos de fornecedor ou estoque.
